# Tutorial about fluopy - add fluorophore and transition data

Here we provide some hints on extending fluoropy with new fluorophore and transition data.

In [2]:
from pathlib import Path

import numpy as np

import fluopy

## Naming conventions

|name|meaning|
|---|---|
|SingleState|photophysical state fo a single fluorophore|
|PairedState|Two SingleStates of paired fluorophores as in donor and acceptor pairs|
|Transition|constant and variable attributes of photophysical transition|
|combined states|combinations of SingleStates depending on number of fluorophores|
|combined state transition|transitions between combined states|
|realizable|theoretically possible|
|resample|change the frame integration time|

## Adding a fluorophore

A fluorophore that is not included with Fluopy can be defined in a notebook.

1. Create an S0 absorption Spectrum from arrays or a CSV file. Also create an
   emission Spectrum when bandpass filtering or energy-transfer calculations are
   required.
2. Create a FluorophoreData object containing the spectra and photophysical constants.
3. Pass the FluorophoreData object to Fluorophore
4. Use FluorophoreSystem.load_transitions() to derive transitions automatically.

In [19]:
wavelengths = np.array([600, 620, 640, 660, 680])

emission = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[0.0, 0.2, 1.0, 0.6, 0.1],
)
absorption_s0 = fluopy.Spectrum.from_arrays(
    wavelengths=wavelengths,
    values=[10000, 40000, 80000, 30000, 5000],
)

custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=emission,
    absorption_spectra={"s0": absorption_s0},
)

custom_fluorophore_1 = fluopy.Fluorophore(
    name="custom",
    position=[0, 0],
    constants=custom_data,
)

custom_fluorophore_2 = fluopy.Fluorophore(
    name="custom",
    position=[0, 1],
    constants=custom_data,
)

fluorophore_system = fluopy.FluorophoreSystem(
    fluorophores=[custom_fluorophore_1, custom_fluorophore_2],
)
transitions = fluorophore_system.load_transitions(
    wavelength=640,
    energy_transfer=False,
    dstorm=False,
)

In [20]:
spectrum_dir = Path(fluopy.__file__).parent / "fluorophore_spectra" / "atto643_data"

custom_data = fluopy.FluorophoreData(
    QUANTUM_YIELD=0.6,
    FLUORESCENCE_LIFETIME=3e-9,
    emission_spectrum=fluopy.Spectrum.from_csv(spectrum_dir / "emission.csv"),
    absorption_spectra={
        "s0": fluopy.Spectrum.from_csv(spectrum_dir / "absorption_s0.csv"),
    },
)

## Adding a single state transition
A transition involving a photophysical state that is not included with Fluopy can
be defined in a notebook.

1. Create a SingleState with a unique name and numerical value (if desired state missing).
2. Create a TransitionType defining the initial state, final state, abbreviation,
   and whether a photon is emitted.
3. Create a Transition containing the transition type, rate, and relevant
   fluorophore identities.
4. Add the transition to the transition dictionary under the fluorophore name.
5. Create a TransitionSet from the updated dictionary and FluorophoreSystem.

FluorophoreSystem.load_transitions() only derives built-in transition types.
Custom transitions therefore have to be added to the resulting transition
dictionary manually.

In [ ]:
dark = fluopy.SingleState(name="DARK", value=10)

recovery = fluopy.TransitionType(
    abbreviation="REC",
    initial_state=dark,
    final_state=fluopy.SingleState.S0,
    photon=False,
)

transitions["custom"].append(
    fluopy.Transition(
        transition_type=recovery,
        rate=1e3,
        fluorophore_ids=[0],
    )
)

transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Adding an ET transition

A custom transition involving two fluorophores can be defined using PairedState
objects.

1. Create any additional SingleStates required by the transition.
2. Create PairedStates describing the donor and acceptor states before and after
   the transition (if missing).
3. Create a TransitionType using the initial and final PairedStates.
4. Create a Transition whose fluorophore identities are donor–acceptor tuples.
5. Add the transition under a key containing the donor name, acceptor name, and
   distance from the FluorophoreSystem.
6. Create a TransitionSet from the updated transition dictionary.

In [13]:
s1_dark = fluopy.PairedState(
    name="S1_DARK",
    donor=fluopy.SingleState.S1,
    acceptor=dark,
)
s0_dark = fluopy.PairedState(
    name="S0_DARK",
    donor=fluopy.SingleState.S0,
    acceptor=dark,
)

custom_et = fluopy.TransitionType(
    abbreviation="DARK_ET",
    initial_state=s1_dark,
    final_state=s0_dark,
    photon=False,
)

donor_id, acceptor_id = 0, 1
donor = fluorophore_system.fluorophores[donor_id]
acceptor = fluorophore_system.fluorophores[acceptor_id]
distance = fluorophore_system.distances[(donor_id, acceptor_id)]

transition_key = f"D: {donor.name}, A: {acceptor.name}, dist: {distance}"

transitions.setdefault(transition_key, []).append(
    fluopy.Transition(
        transition_type=custom_et,
        rate=1e6,
        fluorophore_ids=[(donor_id, acceptor_id)],
    )
)

transition_set = fluopy.TransitionSet(
    transitions=transitions,
    fluorophore_system=fluorophore_system,
)

## Keep in mind

### Spectra
A spectrum consists of one-dimensional wavelength and value arrays. Wavelengths
are given in nm and must be strictly increasing. Spectrum values must be non-negative.

Absorption spectra contain absolute molar extinction coefficients. Emission
spectra may contain relative intensities because they are normalized where
required.

CSV files use 'Wavelengths' and 'y' as the default column names. Alternative
column names can be passed to Spectrum.from_csv().

## Constants
If a cross section is provided, it should correspond to the excitation wavelength used. Energy transfers refer to absorption spectra, not individual cross sections.